In [1]:
# ============================================================
# EXPERIMENT 004
# XGBoost with explicit missingness indicators
# ============================================================

from pathlib import Path
from time import time
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBClassifier


warnings.filterwarnings("ignore")


# ============================================================
# 1. CONFIGURATION
# ============================================================

EXPERIMENT_ID = "EXP-004"

RANDOM_STATE = 42
N_SPLITS = 3

TARGET = "addicted_label"
ID_COLUMN = "id"

XGB_BASELINE_AUC = 0.963034

# Require a small but visible local improvement before submitting.
MINIMUM_IMPROVEMENT = 0.0002
SUBMISSION_THRESHOLD = XGB_BASELINE_AUC + MINIMUM_IMPROVEMENT


# Make paths work from either the project root or notebooks folder.
PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data"
SUBMISSION_DIR = PROJECT_DIR / "submissions"

SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"


# ============================================================
# 2. LOAD DATA
# ============================================================

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print(f"Train shape:             {train.shape}")
print(f"Test shape:              {test.shape}")
print(f"Sample submission shape: {sample_submission.shape}")

assert TARGET in train.columns
assert TARGET not in test.columns
assert len(test) == len(sample_submission)


# ============================================================
# 3. CREATE ORIGINAL FEATURES AND TARGET
# ============================================================

X = train.drop(
    columns=[TARGET, ID_COLUMN]
).copy()

y = train[TARGET].astype(int).copy()

X_test = test.drop(
    columns=[ID_COLUMN]
).copy()

assert list(X.columns) == list(X_test.columns)

original_features = X.columns.tolist()


# ============================================================
# 4. ADD MISSINGNESS INDICATORS
# ============================================================

# Only create indicators for columns that have missing values
# in either the training or test data.

columns_with_missing_values = [
    column
    for column in original_features
    if X[column].isna().any() or X_test[column].isna().any()
]


for column in columns_with_missing_values:
    missing_indicator_name = f"{column}__missing"

    X[missing_indicator_name] = (
        X[column].isna().astype("int8")
    )

    X_test[missing_indicator_name] = (
        X_test[column].isna().astype("int8")
    )


missing_indicator_columns = [
    f"{column}__missing"
    for column in columns_with_missing_values
]


print(f"\nOriginal features:          {len(original_features)}")
print(f"Missingness indicators:     {len(missing_indicator_columns)}")
print(f"Total experiment features:  {X.shape[1]}")

print("\nColumns receiving missingness indicators:")
print(columns_with_missing_values)

print("\nIndicator prevalence in training data:")

indicator_prevalence = (
    X[missing_indicator_columns]
    .mean()
    .sort_values(ascending=False)
)

print(indicator_prevalence.round(4))


# Confirm feature alignment.
assert list(X.columns) == list(X_test.columns)


# ============================================================
# 5. IDENTIFY FEATURE TYPES
# ============================================================

categorical_columns = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

numeric_columns = X.columns.difference(
    categorical_columns
).tolist()


print(f"\nNumerical features:   {len(numeric_columns)}")
print(f"Categorical features: {len(categorical_columns)}")

print("\nCategorical columns:")
print(categorical_columns)


# ============================================================
# 6. PREPROCESSING
# ============================================================

try:
    one_hot_encoder = OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=10,
        sparse_output=True
    )
except TypeError:
    one_hot_encoder = OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=10,
        sparse=True
    )


numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)


categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "one_hot",
            one_hot_encoder
        )
    ]
)


preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_columns
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_columns
        )
    ],
    remainder="drop"
)


# ============================================================
# 7. XGBOOST CONFIGURATION
# ============================================================

# These are intentionally unchanged from EXP-001.

model_parameters = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "n_estimators": 1500,
    "learning_rate": 0.05,
    "max_depth": 6,
    "min_child_weight": 5,
    "subsample": 0.80,
    "colsample_bytree": 0.80,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "tree_method": "hist",
    "early_stopping_rounds": 75,
    "random_state": RANDOM_STATE,
    "n_jobs": -1
}


# ============================================================
# 8. STRATIFIED CROSS-VALIDATION
# ============================================================

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)


out_of_fold_predictions = np.zeros(
    len(train),
    dtype=float
)

test_predictions = np.zeros(
    len(test),
    dtype=float
)

fold_scores = []
best_iterations = []
fold_feature_importances = []

experiment_start = time()


for fold_number, (train_indices, validation_indices) in enumerate(
    cv.split(X, y),
    start=1
):
    fold_start = time()

    X_train_fold = X.iloc[train_indices]
    X_validation_fold = X.iloc[validation_indices]

    y_train_fold = y.iloc[train_indices]
    y_validation_fold = y.iloc[validation_indices]

    # Fit preprocessing only on the current training fold.
    fold_preprocessor = clone(preprocessor)

    X_train_processed = fold_preprocessor.fit_transform(
        X_train_fold
    )

    X_validation_processed = fold_preprocessor.transform(
        X_validation_fold
    )

    X_test_processed = fold_preprocessor.transform(
        X_test
    )

    if fold_number == 1:
        print(
            "\nProcessed training matrix shape:",
            X_train_processed.shape
        )

    model = XGBClassifier(
        **model_parameters
    )

    model.fit(
        X_train_processed,
        y_train_fold,
        eval_set=[
            (
                X_validation_processed,
                y_validation_fold
            )
        ],
        verbose=False
    )

    validation_probabilities = model.predict_proba(
        X_validation_processed
    )[:, 1]

    fold_test_probabilities = model.predict_proba(
        X_test_processed
    )[:, 1]

    out_of_fold_predictions[validation_indices] = (
        validation_probabilities
    )

    test_predictions += (
        fold_test_probabilities / N_SPLITS
    )

    fold_auc = roc_auc_score(
        y_validation_fold,
        validation_probabilities
    )

    fold_scores.append(float(fold_auc))

    best_iteration = getattr(
        model,
        "best_iteration",
        None
    )

    best_iterations.append(best_iteration)

    # Retrieve the final transformed feature names.
    transformed_feature_names = (
        fold_preprocessor.get_feature_names_out()
    )

    fold_importance = pd.Series(
        model.feature_importances_,
        index=transformed_feature_names,
        name=f"fold_{fold_number}"
    )

    fold_feature_importances.append(
        fold_importance
    )

    fold_minutes = (
        time() - fold_start
    ) / 60

    print(
        f"Fold {fold_number}/{N_SPLITS} | "
        f"AUC: {fold_auc:.6f} | "
        f"Best iteration: {best_iteration} | "
        f"Time: {fold_minutes:.2f} minutes"
    )


# ============================================================
# 9. VALIDATION RESULTS
# ============================================================

overall_oof_auc = roc_auc_score(
    y,
    out_of_fold_predictions
)

mean_fold_auc = float(
    np.mean(fold_scores)
)

std_fold_auc = float(
    np.std(fold_scores)
)

difference_vs_baseline = (
    overall_oof_auc - XGB_BASELINE_AUC
)

total_minutes = (
    time() - experiment_start
) / 60


print("\n" + "=" * 60)
print("EXPERIMENT 004 RESULTS")
print("=" * 60)

for fold_number, score in enumerate(
    fold_scores,
    start=1
):
    print(
        f"Fold {fold_number} AUC: "
        f"{score:.6f}"
    )

print(f"\nMean fold AUC:          {mean_fold_auc:.6f}")
print(f"Fold AUC SD:            {std_fold_auc:.6f}")
print(f"OOF AUC:                {overall_oof_auc:.6f}")
print(f"EXP-001 XGBoost OOF:    {XGB_BASELINE_AUC:.6f}")
print(f"Difference vs baseline: {difference_vs_baseline:+.6f}")
print(f"Best iterations:        {best_iterations}")
print(f"Runtime:                {total_minutes:.2f} minutes")


# ============================================================
# 10. INTERPRET RESULT
# ============================================================

print("\nInterpretation:")

if difference_vs_baseline >= MINIMUM_IMPROVEMENT:
    print(
        "Missingness indicators produced a useful improvement. "
        "The pattern of missing values contains predictive signal."
    )

elif difference_vs_baseline > -MINIMUM_IMPROVEMENT:
    print(
        "The result is effectively unchanged. Missingness indicators "
        "provide little additional information."
    )

else:
    print(
        "Missingness indicators reduced validation performance. "
        "They likely add noise or redundant information."
    )


# ============================================================
# 11. FEATURE IMPORTANCE
# ============================================================

feature_importance_table = pd.concat(
    fold_feature_importances,
    axis=1
).fillna(0)

feature_importance_table["mean_importance"] = (
    feature_importance_table.mean(axis=1)
)

feature_importance_table["importance_sd"] = (
    feature_importance_table[
        [
            "fold_1",
            "fold_2",
            "fold_3"
        ]
    ].std(axis=1)
)

feature_importance_table = (
    feature_importance_table
    .sort_values(
        "mean_importance",
        ascending=False
    )
)


print("\nTop 25 transformed features:")
print(
    feature_importance_table[
        [
            "mean_importance",
            "importance_sd"
        ]
    ]
    .head(25)
    .round(6)
)


# Show only missingness indicators.
missing_importance_mask = (
    feature_importance_table.index
    .str.contains("__missing", regex=False)
)

missing_indicator_importance = (
    feature_importance_table.loc[
        missing_importance_mask,
        [
            "mean_importance",
            "importance_sd"
        ]
    ]
    .sort_values(
        "mean_importance",
        ascending=False
    )
)


print("\nMissingness-indicator importance:")
print(
    missing_indicator_importance.round(6)
)


# ============================================================
# 12. PREDICTION SANITY CHECKS
# ============================================================

prediction_summary = pd.Series(
    test_predictions,
    name="predicted_probability"
).describe()

print("\nTest prediction summary:")
print(prediction_summary)


assert np.isfinite(
    out_of_fold_predictions
).all()

assert np.isfinite(
    test_predictions
).all()

assert (
    (test_predictions >= 0) &
    (test_predictions <= 1)
).all()

assert np.std(test_predictions) > 0


# ============================================================
# 13. CONDITIONAL SUBMISSION
# ============================================================

if overall_oof_auc >= SUBMISSION_THRESHOLD:

    submission = sample_submission.copy()

    assert TARGET in submission.columns

    submission[TARGET] = test_predictions

    submission_path = (
        SUBMISSION_DIR /
        "exp_004_xgb_missing_indicators.csv"
    )

    submission.to_csv(
        submission_path,
        index=False
    )

    print(
        "\nSubmission created because EXP-004 beat "
        "the XGBoost baseline by at least "
        f"{MINIMUM_IMPROVEMENT:.4f}."
    )

    print(f"Saved to:\n{submission_path}")

    print("\nSubmission preview:")
    print(submission.head())

else:
    submission_path = None

    print(
        "\nNo submission created because OOF AUC did not reach "
        f"{SUBMISSION_THRESHOLD:.6f}."
    )

Train shape:             (691369, 14)
Test shape:              (296302, 13)
Sample submission shape: (296302, 2)

Original features:          12
Missingness indicators:     12
Total experiment features:  24

Columns receiving missingness indicators:
['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours', 'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time', 'gender', 'stress_level', 'academic_work_impact']

Indicator prevalence in training data:
social_media_hours__missing         0.1938
gaming_hours__missing               0.1834
weekend_screen_time__missing        0.1621
daily_screen_time_hours__missing    0.1386
app_opens_per_day__missing          0.1167
notifications_per_day__missing      0.0978
stress_level__missing               0.0798
work_study_hours__missing           0.0745
sleep_hours__missing                0.0643
academic_work_impact__missing       0.0640
gender__missing                     0.0420
age__missi

### Experiment Log

In [2]:
from datetime import datetime
from pathlib import Path
import json

import numpy as np
import pandas as pd


EXPERIMENT_LOG_PATH = PROJECT_DIR / "experiment_log.csv"


def log_experiment(
    experiment_id,
    description,
    model,
    features,
    validation_method,
    cv_scores,
    kaggle_score=None,
    changes="",
    submission_file="",
    notes="",
    log_path=EXPERIMENT_LOG_PATH
):
    """
    Add or update one experiment in experiment_log.csv.

    If the experiment_id already exists, its previous row is replaced.
    """

    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)

    cv_scores = [float(score) for score in cv_scores]

    cv_mean = float(np.mean(cv_scores))
    cv_std = float(np.std(cv_scores))

    kaggle_score_value = (
        float(kaggle_score)
        if kaggle_score is not None
        else np.nan
    )

    kaggle_cv_gap = (
        kaggle_score_value - cv_mean
        if pd.notna(kaggle_score_value)
        else np.nan
    )

    experiment_record = {
        "experiment_id": experiment_id,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "description": description,
        "model": model,
        "features": json.dumps(list(features)),
        "n_features": len(features),
        "validation_method": validation_method,
        "cv_scores": json.dumps(cv_scores),
        "cv_mean": cv_mean,
        "cv_std": cv_std,
        "kaggle_score": kaggle_score_value,
        "kaggle_cv_gap": kaggle_cv_gap,
        "changes": changes,
        "submission_file": submission_file,
        "notes": notes
    }

    if log_path.exists():
        experiments = pd.read_csv(log_path)

        # Prevent duplicate rows when rerunning the same experiment cell.
        if "experiment_id" in experiments.columns:
            experiments = experiments[
                experiments["experiment_id"] != experiment_id
            ].copy()
    else:
        experiments = pd.DataFrame()

    new_row = pd.DataFrame([experiment_record])

    experiments = pd.concat(
        [experiments, new_row],
        ignore_index=True
    )

    experiments = experiments.sort_values(
        by="experiment_id"
    ).reset_index(drop=True)

    experiments.to_csv(log_path, index=False)

    print(f"Logged {experiment_id}")
    print(f"CV mean:       {cv_mean:.6f}")
    print(f"CV SD:         {cv_std:.6f}")

    if pd.notna(kaggle_score_value):
        print(f"Kaggle score:  {kaggle_score_value:.6f}")
        print(f"Kaggle-CV gap: {kaggle_cv_gap:+.6f}")

    print(f"Log saved to:  {log_path}")

    return experiments

In [3]:
experiments = log_experiment(
    experiment_id="EXP-004",
    description=(
        "XGBoost baseline extended with binary missingness indicators "
        "for every feature containing missing values."
    ),
    model="XGBClassifier",
    features=X.columns.tolist(),
    validation_method="3-fold StratifiedKFold with ROC AUC",
    cv_scores=[
        0.962723,
        0.963675,
        0.963473
    ],
    kaggle_score=0.96475,
    changes=(
        "Added one binary missingness indicator for each feature with "
        "missing values while keeping the EXP-001 preprocessing, model "
        "parameters, folds, and original features unchanged."
    ),
    submission_file="exp_004_xgb_missing_indicators.csv",
    notes=(
        "OOF AUC was 0.963290 with fold SD 0.000409. This improved upon "
        "EXP-001 by 0.000256. Kaggle improved from 0.964490 to 0.964750, "
        "a gain of 0.000260 that closely matched the local CV improvement. "
        "All folds reached best iteration 1499, suggesting the estimator "
        "limit may still be constraining the model."
    )
)

experiments.tail()

Logged EXP-004
CV mean:       0.963290
CV SD:         0.000410
Kaggle score:  0.964750
Kaggle-CV gap: +0.001460
Log saved to:  C:\Users\Owner\Documents\Github\machine-learning-lab\00-Kaggle\02-smartphone-addiction\experiment_log.csv


,experiment_id,timestamp,description,model,features,n_features,validation_method,cv_scores,cv_mean,cv_std,kaggle_score,kaggle_cv_gap,changes,submission_file,notes
0,EXP-001,2026-08-02 21:49:55,Initial XGBoost baseline using median-imputed ...,XGBClassifier,"[""age"", ""daily_screen_time_hours"", ""social_med...",12,3-fold StratifiedKFold with ROC AUC,"[0.962477, 0.963382, 0.963244]",0.963034,0.000398,0.96449,0.001456,Established the first end-to-end baseline usin...,exp_001_xgb_baseline.csv,OOF AUC was 0.963034 with fold SD 0.000398. Ka...
1,EXP-002,2026-08-02 21:53:06,Logistic-regression control using standardized...,LogisticRegression,"[""age"", ""daily_screen_time_hours"", ""social_med...",12,3-fold StratifiedKFold with ROC AUC,"[0.910347, 0.911866, 0.912106]",0.911440,0.000779,NaN,NaN,Replaced the XGBoost baseline with a regulariz...,NaN,OOF AUC was 0.911437 with fold SD 0.000779. Th...
2,EXP-003,2026-08-02 22:11:48,CatBoost comparison using native numerical mis...,CatBoostClassifier,"[""age"", ""daily_screen_time_hours"", ""social_med...",12,3-fold StratifiedKFold with ROC AUC,"[0.961586, 0.962336, 0.962426]",0.962116,0.000377,NaN,NaN,Replaced XGBoost and one-hot preprocessing wit...,NaN,OOF AUC was 0.962115 with fold SD 0.000376. Ca...
3,EXP-004,2026-08-02 22:18:27,XGBoost baseline extended with binary missingn...,XGBClassifier,"[""age"", ""daily_screen_time_hours"", ""social_med...",24,3-fold StratifiedKFold with ROC AUC,"[0.962723, 0.963675, 0.963473]",0.963290,0.000410,0.96475,0.001460,Added one binary missingness indicator for eac...,exp_004_xgb_missing_indicators.csv,OOF AUC was 0.963290 with fold SD 0.000409. Th...
